# 🚀 Milli-Scrapes Cloud Evaluation & Visual Perception Gauntlet

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-Al-Raju/milli-scrapes/blob/main/notebooks/milli_scrapes_colab_eval.ipynb)

> **Zero-Footprint Cloud Offloading:** Run massive visual extraction missions (2,000+ images), high-concurrency lead scraping (10,000 leads), and real-time DOM settling on Google Cloud hardware without burdening your local machine.
>
> **Default Visual Grounding:** Integrates `MilliVisualVerifier` & `MilliEye` perception to eliminate keyword hallucinations (e.g. automatically rejecting cakes, biscuits, shoes, and non-target objects).

### 1. Environment Setup
Clones the latest Milli-Scrapes engine, sets up high-performance Chromium dependencies, and initializes hardware perception.

In [ ]:
#@title ⚡ Install Milli-Scrapes & Cloud Dependencies
!git clone https://github.com/Abdullah-Al-Raju/milli-scrapes.git /content/milli-scrapes 2>/dev/null || (cd /content/milli-scrapes && git pull)
%cd /content/milli-scrapes
!pip install -q -r requirements.txt
!pip install -q httpx pillow numpy psutil beautifulsoup4

import os, sys, psutil
print(f"✓ Milli-Scrapes Engine loaded! Available Cloud RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")

### 2. Configure Your Mission
Select whether to run **Bulk Visual Extraction with Anti-Hallucination** (e.g., 2,000 Verified Images) or **10,000 Enterprise Lead Showdown**.

In [ ]:
#@title 🎯 Mission Parameters
MISSION_TYPE = "Visual Image Extraction" #@param ["Visual Image Extraction", "10,000 Leads Benchmark", "Real Website Conflict Audit"]
TARGET_QUERY = "Selena Gomez" #@param {type:"string"}
TARGET_COUNT = 2000 #@param {type:"integer"}
VISUAL_VERIFICATION = True #@param {type:"boolean"}
TARGET_CATEGORY = "portrait" #@param ["portrait", "table", "document", "general"]
CONCURRENCY = 30 #@param {type:"integer"}

print(f"Configured Mission: {MISSION_TYPE}")
print(f"Target: {TARGET_QUERY} | Target Count: {TARGET_COUNT} | Visual Verification: {VISUAL_VERIFICATION}")

### 3. Execute Mission on Google Cloud
Streams assets concurrently, evaluates every frame with `MilliVisualVerifier` to discard non-target objects (cakes, biscuits, logos), and logs empirical throughput.

In [ ]:
#@title ⚡ Run Milli Cloud Pipeline
import asyncio, time, io, urllib.parse, httpx, zipfile
from PIL import Image
from milli_scrapes import UniversalScraper, MilliVisualVerifier, VisualCategory

OUTPUT_DIR = "/content/milli_output/dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

async def run_cloud_mission():
    t0_global = time.perf_counter()
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"}
    
    if MISSION_TYPE == "10,000 Leads Benchmark":
        print(">>> Running 10,000 Enterprise Leads Benchmark Showdown...")
        !python3 benchmarks/benchmark_10000_leads_scale_showdown.py
        !cp -r BENCHMARK_10000_LEADS_REPORT.md /content/milli_output/
        !cp -r benchmarks/benchmark_10000_leads_results.json /content/milli_output/
        return

    print(f">>> [Phase 1] Discovering image targets for: '{TARGET_QUERY}'...")
    scraper = UniversalScraper(headless=True, prefer_fast_path=True, auto_consent=True)
    
    # Collect candidate queries
    subqueries = [TARGET_QUERY, f"{TARGET_QUERY} portrait", f"{TARGET_QUERY} concert", f"{TARGET_QUERY} 2024", f"{TARGET_QUERY} event", f"{TARGET_QUERY} red carpet"]
    candidate_titles = set()
    
    async with httpx.AsyncClient(headers=headers, timeout=20.0, follow_redirects=True) as client:
        for q in subqueries:
            for offset in (0, 500, 1000):
                url = f"https://commons.wikimedia.org/w/api.php?action=query&list=search&srsearch={urllib.parse.quote(q)}&srnamespace=6&srlimit=500&sroffset={offset}&format=json"
                try:
                    r = await client.get(url)
                    if r.status_code == 200:
                        hits = r.json().get("query", {}).get("search", [])
                        if not hits: break
                        for h in hits:
                            t = h.get("title", "")
                            if any(t.lower().endswith(ext) for ext in (".jpg", ".jpeg", ".png", ".webp")):
                                candidate_titles.add(t)
                except Exception:
                    pass
                if len(candidate_titles) >= TARGET_COUNT + 300:
                    break
            if len(candidate_titles) >= TARGET_COUNT + 300:
                break
                
        title_list = list(candidate_titles)
        print(f"✓ Discovered {len(title_list)} candidate asset titles on the web.")
        
        # Batch resolve URLs via POST
        print(">>> [Phase 2] Batch resolving direct media endpoints...")
        resolved_assets = []
        for i in range(0, len(title_list), 50):
            chunk = title_list[i:i+50]
            post_data = {"action": "query", "titles": "|".join(chunk), "prop": "imageinfo", "iiprop": "url|size|mime", "iiurlwidth": "640", "format": "json"}
            try:
                r_post = await client.post("https://commons.wikimedia.org/w/api.php", data=post_data)
                if r_post.status_code == 200:
                    pages = r_post.json().get("query", {}).get("pages", {})
                    for pid, p in pages.items():
                        if "imageinfo" in p:
                            u = p["imageinfo"][0].get("thumburl") or p["imageinfo"][0].get("url")
                            if u: resolved_assets.append({"title": p.get("title", ""), "url": u})
            except Exception:
                pass
                
        print(f"✓ Resolved {len(resolved_assets)} streaming endpoints.")
        
        # Phase 3: Download & Visual Verification
        print(f">>> [Phase 3] Streaming & Visually Verifying {TARGET_COUNT:,} assets (Rejecting cakes/objects)...")
        sem = asyncio.Semaphore(CONCURRENCY)
        verified_items = []
        rejected_count = 0
        counter = 0
        lock = asyncio.Lock()
        t0_dl = time.perf_counter()
        
        async def process_one(asset):
            nonlocal counter, rejected_count
            async with sem:
                if len(verified_items) >= TARGET_COUNT:
                    return
                try:
                    res = await client.get(asset["url"])
                    if res.status_code != 200:
                        return
                    content = res.content
                except Exception:
                    return
                    
                # Visual Verification Filter
                if VISUAL_VERIFICATION:
                    v_res = MilliVisualVerifier.verify_buffer(content, target_category=VisualCategory(TARGET_CATEGORY))
                    if not v_res.is_valid:
                        async with lock:
                            rejected_count += 1
                        return
                        
                async with lock:
                    if len(verified_items) >= TARGET_COUNT:
                        return
                    counter += 1
                    idx = counter
                    ext = "png" if content.startswith(b"\x89PNG") else "jpg"
                    raw_name = f"image-{idx:04d}"
                    file_name = f"{raw_name}.{ext}"
                    p_ext = os.path.join(OUTPUT_DIR, file_name)
                    p_raw = os.path.join(OUTPUT_DIR, raw_name)
                    with open(p_ext, "wb") as f: f.write(content)
                    if not os.path.exists(p_raw):
                        try: os.link(p_ext, p_raw)
                        except Exception: 
                            with open(p_raw, "wb") as f: f.write(content)
                            
                    verified_items.append({"index": idx, "file": file_name, "bytes": len(content)})
                    if idx % 100 == 0 or idx == TARGET_COUNT:
                        now_el = time.perf_counter() - t0_dl
                        spd = idx / now_el if now_el > 0 else 0
                        print(f"  • Verified {idx:4d} / {TARGET_COUNT} | Speed: {spd:5.1f} imgs/sec | Non-target Rejected: {rejected_count}")

        await asyncio.gather(*(process_one(a) for a in resolved_assets))
        
    wall_time = time.perf_counter() - t0_dl
    total_bytes = sum(x["bytes"] for x in verified_items)
    print("=" * 75)
    print(f"MISSION COMPLETE!")
    print(f"✓ Verified Saved: {len(verified_items):,} assets ({total_bytes / (1024*1024):.2f} MB)")
    print(f"✓ Non-Target Hallucinations Rejected (Cakes/Biscuits/Objects): {rejected_count}")
    print(f"✓ Wall-Clock Time: {wall_time:.2f}s (Average Speed: {len(verified_items)/wall_time:.1f} imgs/sec)")
    print("=" * 75)

await run_cloud_mission()

### 4. Package, Download & Auto-Extract Locally
Compresses all verified dataset files and benchmark reports into `milli_scrapes_results.zip`, triggers direct browser download, and displays the command to extract into your local folder.

In [ ]:
#@title 📦 Download Results Archive (.zip)
from google.colab import files
import shutil

zip_path = "/content/milli_scrapes_results.zip"
print(">>> Packaging verified assets and reports...")
shutil.make_archive("/content/milli_scrapes_results", "zip", "/content/milli_output")

print(f"✓ Created archive: {zip_path} ({os.path.getsize(zip_path) / (1024*1024):.2f} MB)")
print(">>> Initiating automated browser download...")
files.download(zip_path)

print("\n" + "=" * 75)
print("To extract the downloaded zip into your local milli-scrapes project folder, run:")
print("  unzip -o ~/Downloads/milli_scrapes_results.zip -d output/")
print("=" * 75)